# Generative Models Overview

## Learning Objectives
- Understand different types of generative models.
- Learn the uses and challenges of generative modeling.

## Introduction
Generative AI refers to a category of AI models that can create new content, including text, images, audio, and more, based on the data they've been trained on. Unlike traditional AI focused on analysis and prediction, generative AI actively produces novel outputs. This is achieved through algorithms like Generative Adversarial Networks (GANs) and large language models (LLMs), which learn patterns from data and then generate new, similar content. 

Generative AI models are trained on vast datasets of information (text, images, audio, etc.) and then use that knowledge to create new content. They don't just identify patterns; they actively generate new, original pieces. 

Training:
Models learn from massive datasets, identifying patterns and relationships within the data.

Generation:
When prompted, these models leverage their learned patterns to create new content. For example, an LLM trained on text might generate a poem or an article, while a GAN might create a new image. 

Generative models learn to create new data that resembles the training data, with applications in images, text, and more.

## Core Types
- **Generative Adversarial Networks (GANs):** Adversarial training of generator and discriminator.
- **Variational Autoencoders (VAEs):** Probabilistic models learning latent variables.
- **Autoregressive Models:** Sequential generation of data points.

## Example
Basic VAE architecture outline.

In [7]:
import tensorflow as tf
from tensorflow.keras import layers

class Sampling(layers.Layer):
    """Uses (z_mean, z_log_var) to sample z, the vector encoding a digit."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        # The shape of z_mean is (batch_size, latent_dim), so dim should be the second element.
        dim = tf.shape(z_mean)[1] 
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

latent_dim = 2

# Encoder
encoder_inputs = layers.Input(shape=(28, 28, 1))
x = layers.Flatten()(encoder_inputs)
x = layers.Dense(64, activation='relu')(x)
z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])
encoder = tf.keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")
encoder.summary()

Model: "encoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_10      │ (None, 28, 28, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_6 (Flatten) │ (None, 784)       │          0 │ input_layer_10[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 64)        │     50,240 │ flatten_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_mean (Dense)      │ (None, 2)         │        130 │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_log_var (Dense)   │ (None, 2)         │        130 │ dense_16[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sampling_6          │ (None, 2)         │          0 │ z_mean[0][0],     │
│ (Sampling)          │                   │            │ z_log_var[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 50,500 (197.27 KB)

 Trainable params: 50,500 (197.27 KB)

 Non-trainable params: 0 (0.00 B)

Complete VAE pipeline in Tensorflow

In [2]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

# --- 1. ENCODER and DECODER (No changes here) ---
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

latent_dim = 2
encoder_inputs = layers.Input(shape=(28, 28, 1))
x = layers.Flatten()(encoder_inputs)
x = layers.Dense(128, activation='relu')(x)
z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])
encoder = tf.keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

latent_inputs = layers.Input(shape=(latent_dim,))
x = layers.Dense(128, activation='relu')(latent_inputs)
x = layers.Dense(28 * 28, activation='sigmoid')(x)
decoder_outputs = layers.Reshape((28, 28, 1))(x)
decoder = tf.keras.Model(latent_inputs, decoder_outputs, name="decoder")


# --- 2. REWRITTEN VAE MODEL ---
# This version uses `self.add_loss` instead of a custom train_step
class VAE(tf.keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def call(self, inputs):
        # Get the latent vector and its distribution parameters from the encoder
        z_mean, z_log_var, z = self.encoder(inputs)
        
        # Reconstruct the image using the decoder
        reconstruction = self.decoder(z)
        
        # Calculate the KL divergence loss
        kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
        
        # Add the KL loss to the model's main loss
        self.add_loss(kl_loss)
        
        return reconstruction


# --- 3. DATA PREPARATION & TRAINING (with new compile and fit) ---
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()
mnist_digits = np.concatenate([x_train, x_test], axis=0)
mnist_digits = np.expand_dims(mnist_digits, -1).astype("float32") / 255

# Instantiate the VAE
vae = VAE(encoder, decoder)

# Compile the model specifying only the reconstruction loss.
# The KL loss is already added inside the model.
vae.compile(optimizer=tf.keras.optimizers.Adam(), loss=tf.keras.losses.MeanSquaredError())

print("\n--- Starting VAE Training (New Method) ---")
# The model expects both input (x) and target (y). For an autoencoder, they are the same.
vae.fit(mnist_digits, mnist_digits, epochs=30, batch_size=128)
print("\n--- VAE Training Complete ---")

2025-09-14 16:16:22.827447: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2025-09-14 16:16:22.827610: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2025-09-14 16:16:22.827614: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 10.67 GB
I0000 00:00:1757862982.827887  338815 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1757862982.828320  338815 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)



--- Starting VAE Training (New Method) ---
Epoch 1/30


2025-09-14 16:16:23.809957: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


547/547 ━━━━━━━━━━━━━━━━━━━━ 12s 19ms/step - loss: 0.1063
Epoch 2/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - loss: 0.0684
Epoch 3/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - loss: 0.0675
Epoch 4/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - loss: 0.0675
Epoch 5/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - loss: 0.0675
Epoch 6/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - loss: 0.0675
Epoch 7/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - loss: 0.0675
Epoch 8/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - loss: 0.0675
Epoch 9/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - loss: 0.0675
Epoch 10/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - loss: 0.0675
Epoch 11/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - loss: 0.0675
Epoch 12/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - loss: 0.0675
Epoch 13/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - loss: 0.0675
Epoch 14/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 10s 19ms/step - loss: 0.0675
Epoch 15/30
547/547 ━━━━━━━━━━━━━━━━━━━━

## Exercise
Rewrite the decoder and VAE model training pipeline with another example training set.

In [1]:
# Write your code here.

## Summary
- Generative models synthesize new data resembling the training set.
- VAEs and GANs are foundational architectures with distinct approaches.


## Further Reading
- [Tutorial on VAEs](https://keras.io/examples/generative/vae/)
- [GANs Introduction](https://arxiv.org/abs/1406.2661)
